# Stock Pre-Selection for New Model Spillovers Pilot

**Objective:** Form 2 stock pairs + 2 familiar anchors for the pilot.

### Pair criteria
- **Within each pair:** similar 12-month price returns, similar market cap, low familiarity priors
- **Between pairs:** heterogeneous P/B percentile (one Low, one High valuation)
- **Pair 1 ("Losers"):** ~−20% to −30% returns over last 365 days
- **Pair 2 ("Winners"):** ~+50% to +70% returns over last 365 days
- **Universe:** Tech sector (GICS sector 45), positive P/B only

### Pipeline
1. Load WRDS Compustat P/B data → compute sector-level percentiles
2. Pull 365-day price history + fundamentals from Yahoo Finance
3. Filter by return ranges, generate candidate pairs
4. Score pairs by valuation contrast, return closeness, market cap similarity, chart shape similarity
5. Output ranked candidates for manual review

In [1]:
import csv
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime, timedelta
from scipy.spatial.distance import euclidean

# Optional: DTW for chart similarity
try:
    from dtaidistance import dtw
    HAS_DTW = True
    print("dtaidistance available — will use DTW for chart similarity")
except ImportError:
    HAS_DTW = False
    print("dtaidistance not installed — will use correlation-based chart similarity")
    print("  Install with: pip install dtaidistance")

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:.2f}'.format)

dtaidistance not installed — will use correlation-based chart similarity
  Install with: pip install dtaidistance


## 1. Load WRDS P/B Data & Compute Sector Percentiles

In [ ]:
# ── Config ──
WRDS_CSV = "/Users/paulgrass/Library/Mobile Documents/com~apple~CloudDocs/Documents/Programming/Git/pilot3-asset-data/WRDS/pb_ratio_sectors_40_45_with_industry.csv"
GSECTOR_IT = "45"  # Information Technology

# ── Load WRDS data, filter to IT sector, positive P/B ──
wrds_raw = pd.read_csv(WRDS_CSV)
wrds = wrds_raw[wrds_raw['gsector'].astype(str) == GSECTOR_IT].copy()
wrds['pb_ratio'] = pd.to_numeric(wrds['pb_ratio'], errors='coerce')
wrds = wrds[wrds['pb_ratio'] > 0].copy()
wrds['tic'] = wrds['tic'].str.strip().str.upper()

# Keep latest entry per ticker
wrds = wrds.sort_values('datadate').drop_duplicates('tic', keep='last')

# ── Compute sector-level P/B percentile (excluding self) ──
all_pbs = wrds['pb_ratio'].values

def pb_percentile_excl_self(row):
    pb = row['pb_ratio']
    peers = all_pbs[all_pbs != pb]  # approximate excl-self
    if len(peers) == 0:
        return np.nan
    return round(100 * np.sum(peers < pb) / len(peers), 1)

wrds['pb_pctile_sector'] = wrds.apply(pb_percentile_excl_self, axis=1)

# Also compute industry-level percentile (gind)
def pb_percentile_industry(row):
    pb = row['pb_ratio']
    ind = row['gind']
    ind_pbs = wrds.loc[(wrds['gind'] == ind) & (wrds['tic'] != row['tic']), 'pb_ratio'].values
    if len(ind_pbs) == 0:
        return np.nan
    return round(100 * np.sum(ind_pbs < pb) / len(ind_pbs), 1)

wrds['pb_pctile_industry'] = wrds.apply(pb_percentile_industry, axis=1)

print(f"IT sector stocks with P/B > 0: {len(wrds)}")
print(f"\nP/B distribution (sector level):")
print(wrds['pb_ratio'].describe().round(2))
print(f"\nIndustry groups (gind):")
print(wrds.groupby('gind').size().sort_values(ascending=False).head(10))

## 2. Fetch Yahoo Finance Data (Prices, Fundamentals)

In [ ]:
# ── Get list of tickers to fetch ──
tickers_to_fetch = wrds['tic'].unique().tolist()
print(f"Fetching data for {len(tickers_to_fetch)} IT-sector tickers...")

# ── Batch download 400-day price history ──
price_data = yf.download(
    tickers_to_fetch,
    period="400d",
    interval="1d",
    auto_adjust=False,
    group_by='ticker',
    threads=True
)
print(f"\nDownloaded price data shape: {price_data.shape}")

In [ ]:
# ── Compute 365-day price-only returns ──
today = pd.Timestamp.today().normalize()
year_ago = today - pd.Timedelta(days=365)

returns_list = []
daily_series = {}  # store normalized daily series for chart similarity

for t in tickers_to_fetch:
    try:
        d = price_data[t][['Close']].dropna().reset_index()
        if len(d) < 100:  # skip if too few data points
            continue
        
        # Current price and 365d-ago price
        p_now = d.iloc[-1]['Close']
        d_start = d[d['Date'] >= year_ago]
        if d_start.empty:
            continue
        p_start = d_start.iloc[0]['Close']
        ret_365d = (p_now / p_start - 1) * 100
        
        returns_list.append({
            'tic': t,
            'price_current': float(p_now),
            'price_365d_ago': float(p_start),
            'ret_365d_pct': float(ret_365d),
            'n_trading_days': len(d)
        })
        
        # Store normalized return series (for chart similarity later)
        recent = d[d['Date'] >= year_ago].copy()
        if len(recent) >= 200:
            recent['norm_ret'] = recent['Close'] / recent['Close'].iloc[0] - 1
            daily_series[t] = recent[['Date', 'norm_ret', 'Close']].reset_index(drop=True)
    except Exception:
        continue

ret_df = pd.DataFrame(returns_list)
print(f"Successfully computed 365d returns for {len(ret_df)} tickers")
print(f"Stored daily series for chart similarity: {len(daily_series)} tickers")

In [ ]:
# ── Fetch fundamentals (market cap, dividend yield) ──
# This takes a while — fetches one by one from yfinance
from tqdm.notebook import tqdm

fund_list = []
tickers_with_returns = ret_df['tic'].tolist()

for t in tqdm(tickers_with_returns, desc="Fetching fundamentals"):
    try:
        info = yf.Ticker(t).info
        fund_list.append({
            'tic': t,
            'company_name': info.get('longName') or info.get('shortName', ''),
            'marketcap': info.get('marketCap'),
            'div_yield_pct': round(info.get('dividendYield', 0) * 100, 2) if info.get('dividendYield') else 0.0,
            'sector_yf': info.get('sector', ''),
            'industry_yf': info.get('industry', ''),
        })
    except Exception:
        fund_list.append({'tic': t, 'company_name': '', 'marketcap': None,
                         'div_yield_pct': None, 'sector_yf': '', 'industry_yf': ''})

fund_df = pd.DataFrame(fund_list)
print(f"Fetched fundamentals for {fund_df['marketcap'].notna().sum()} / {len(fund_df)} tickers")

## 3. Merge & Build Master Table

In [ ]:
# ── Merge WRDS + returns + fundamentals ──
master = (
    wrds[['tic', 'conm', 'gind', 'gsubind', 'pb_ratio', 'pb_pctile_sector', 'pb_pctile_industry']]
    .merge(ret_df, on='tic', how='inner')
    .merge(fund_df, on='tic', how='left')
)

# Add derived columns
master['marketcap_bn'] = (master['marketcap'] / 1e9).round(2)
master['log_mcap'] = np.log(master['marketcap'].replace(0, np.nan))

# Valuation labels based on sector percentile
master['val_sector'] = pd.cut(
    master['pb_pctile_sector'], 
    bins=[-1, 33, 66, 101], 
    labels=['Low', 'Mid', 'High']
)
master['val_industry'] = pd.cut(
    master['pb_pctile_industry'], 
    bins=[-1, 33, 66, 101], 
    labels=['Low', 'Mid', 'High']
)

print(f"Master table: {len(master)} stocks")
print(f"\nReturn distribution:")
print(master['ret_365d_pct'].describe().round(1))
print(f"\nValuation split (sector percentile):")
print(master['val_sector'].value_counts())
print(f"\nMarket cap distribution (billions):")
print(master['marketcap_bn'].describe().round(1))

## 4. Filter to Target Return Ranges

In [ ]:
# ── Return range config (adjustable) ──
LOSER_RANGE  = (-35, -15)   # target: -20% to -30%, with buffer
WINNER_RANGE = (40, 80)     # target: +50% to +70%, with buffer

losers  = master[(master['ret_365d_pct'] >= LOSER_RANGE[0]) & (master['ret_365d_pct'] <= LOSER_RANGE[1])].copy()
winners = master[(master['ret_365d_pct'] >= WINNER_RANGE[0]) & (master['ret_365d_pct'] <= WINNER_RANGE[1])].copy()

print(f"=== LOSERS ({LOSER_RANGE[0]}% to {LOSER_RANGE[1]}%) ===")
print(f"Total: {len(losers)} stocks")
print(f"Low val: {(losers['val_sector']=='Low').sum()}, Mid: {(losers['val_sector']=='Mid').sum()}, High: {(losers['val_sector']=='High').sum()}")
print()

cols_show = ['tic', 'company_name', 'industry_yf', 'ret_365d_pct', 'pb_ratio', 
             'pb_pctile_sector', 'pb_pctile_industry', 'val_sector', 'marketcap_bn', 'div_yield_pct']

print("Losers — Low Valuation:")
display(losers[losers['val_sector']=='Low'][cols_show].sort_values('ret_365d_pct'))

print("\nLosers — High Valuation:")
display(losers[losers['val_sector']=='High'][cols_show].sort_values('ret_365d_pct'))

In [ ]:
print(f"=== WINNERS ({WINNER_RANGE[0]}% to {WINNER_RANGE[1]}%) ===")
print(f"Total: {len(winners)} stocks")
print(f"Low val: {(winners['val_sector']=='Low').sum()}, Mid: {(winners['val_sector']=='Mid').sum()}, High: {(winners['val_sector']=='High').sum()}")
print()

print("Winners — Low Valuation:")
display(winners[winners['val_sector']=='Low'][cols_show].sort_values('ret_365d_pct'))

print("\nWinners — High Valuation:")
display(winners[winners['val_sector']=='High'][cols_show].sort_values('ret_365d_pct'))

## 5. Chart Similarity Measure

In [ ]:
def compute_chart_similarity(tic1, tic2, daily_series_dict):
    """
    Compute chart similarity between two stocks.
    Returns dict with multiple similarity measures:
    - corr: Pearson correlation of normalized return series
    - euclidean: Euclidean distance of normalized returns (lower = more similar)
    - dtw_dist: DTW distance (if dtaidistance available)
    """
    if tic1 not in daily_series_dict or tic2 not in daily_series_dict:
        return {'corr': np.nan, 'euclidean': np.nan, 'dtw_dist': np.nan}
    
    s1 = daily_series_dict[tic1]['norm_ret'].values
    s2 = daily_series_dict[tic2]['norm_ret'].values
    
    # Align to same length (take min)
    n = min(len(s1), len(s2))
    # Resample both to exactly n points via linear interpolation
    s1_r = np.interp(np.linspace(0, 1, n), np.linspace(0, 1, len(s1)), s1)
    s2_r = np.interp(np.linspace(0, 1, n), np.linspace(0, 1, len(s2)), s2)
    
    result = {}
    
    # Pearson correlation
    if np.std(s1_r) > 0 and np.std(s2_r) > 0:
        result['corr'] = float(np.corrcoef(s1_r, s2_r)[0, 1])
    else:
        result['corr'] = np.nan
    
    # Euclidean distance of normalized series
    result['euclidean'] = float(euclidean(s1_r, s2_r))
    
    # DTW distance
    if HAS_DTW:
        try:
            result['dtw_dist'] = float(dtw.distance(s1_r.astype(np.double), s2_r.astype(np.double)))
        except Exception:
            result['dtw_dist'] = np.nan
    else:
        result['dtw_dist'] = np.nan
    
    return result

# Quick test
if len(daily_series) >= 2:
    test_tickers = list(daily_series.keys())[:2]
    test_sim = compute_chart_similarity(test_tickers[0], test_tickers[1], daily_series)
    print(f"Test similarity {test_tickers[0]} vs {test_tickers[1]}: {test_sim}")

## 6. Generate Candidate Pairs

In [ ]:
def generate_pairs(pool, label, max_ret_diff=5.0, min_val_contrast=25, 
                   pctile_col='pb_pctile_sector', top_n=30):
    """
    Generate ranked candidate pairs from a pool of stocks.
    Pairs one Low-valuation stock with one High-valuation stock.
    
    Scoring: higher = better pair
      + valuation contrast (P/B percentile difference)
      - return difference penalty
      - market cap difference penalty (log scale)
      + chart similarity bonus
    """
    low = pool[pool[pctile_col] <= 33].copy()
    high = pool[pool[pctile_col] >= 67].copy()
    
    if low.empty or high.empty:
        # Fallback: use median split
        median_pctile = pool[pctile_col].median()
        low = pool[pool[pctile_col] <= median_pctile].copy()
        high = pool[pool[pctile_col] > median_pctile].copy()
        print(f"  [{label}] Using median split at {median_pctile:.0f}th pctile (Low: {len(low)}, High: {len(high)})")
    else:
        print(f"  [{label}] Low val (<=33rd): {len(low)}, High val (>=67th): {len(high)}")
    
    pairs = []
    for _, r_lo in low.iterrows():
        for _, r_hi in high.iterrows():
            ret_diff = abs(r_lo['ret_365d_pct'] - r_hi['ret_365d_pct'])
            if ret_diff > max_ret_diff:
                continue
            
            val_contrast = abs(r_hi[pctile_col] - r_lo[pctile_col])
            if val_contrast < min_val_contrast:
                continue
            
            # Market cap similarity (log ratio, lower = better)
            mcap_lo = r_lo.get('log_mcap', np.nan)
            mcap_hi = r_hi.get('log_mcap', np.nan)
            mcap_diff = abs(mcap_lo - mcap_hi) if pd.notna(mcap_lo) and pd.notna(mcap_hi) else 5.0
            
            # Chart similarity
            chart_sim = compute_chart_similarity(r_lo['tic'], r_hi['tic'], daily_series)
            chart_corr = chart_sim.get('corr', 0) if pd.notna(chart_sim.get('corr')) else 0
            
            # Composite score
            score = (
                3.0 * (val_contrast / 100)      # reward valuation contrast (0-1)
                - 2.0 * (ret_diff / max_ret_diff) # penalize return diff
                - 1.0 * (mcap_diff / 5.0)        # penalize mcap diff (log units)
                + 1.0 * max(chart_corr, 0)        # reward positive chart correlation
            )
            
            pairs.append({
                'tic_low': r_lo['tic'],
                'name_low': r_lo.get('company_name', r_lo.get('conm', '')),
                'industry_low': r_lo.get('industry_yf', ''),
                'ret_low': r_lo['ret_365d_pct'],
                'pb_pctile_low': r_lo[pctile_col],
                'mcap_bn_low': r_lo.get('marketcap_bn', np.nan),
                'div_low': r_lo.get('div_yield_pct', 0),
                
                'tic_high': r_hi['tic'],
                'name_high': r_hi.get('company_name', r_hi.get('conm', '')),
                'industry_high': r_hi.get('industry_yf', ''),
                'ret_high': r_hi['ret_365d_pct'],
                'pb_pctile_high': r_hi[pctile_col],
                'mcap_bn_high': r_hi.get('marketcap_bn', np.nan),
                'div_high': r_hi.get('div_yield_pct', 0),
                
                'ret_diff_pp': round(ret_diff, 2),
                'val_contrast': round(val_contrast, 1),
                'mcap_diff_log': round(mcap_diff, 2),
                'chart_corr': round(chart_corr, 3),
                'chart_eucl': round(chart_sim.get('euclidean', np.nan), 3) if pd.notna(chart_sim.get('euclidean')) else np.nan,
                'chart_dtw': round(chart_sim.get('dtw_dist', np.nan), 3) if pd.notna(chart_sim.get('dtw_dist')) else np.nan,
                'score': round(score, 3),
            })
    
    if not pairs:
        print(f"  [{label}] No pairs found! Try widening max_ret_diff or lowering min_val_contrast.")
        return pd.DataFrame()
    
    df_pairs = pd.DataFrame(pairs).sort_values('score', ascending=False).head(top_n)
    print(f"  [{label}] Generated {len(pairs)} pairs, showing top {min(top_n, len(pairs))}")
    return df_pairs.reset_index(drop=True)


# ── Generate pairs for both return ranges ──
print("LOSER PAIRS:")
loser_pairs = generate_pairs(
    losers, "Losers", 
    max_ret_diff=8.0,      # allow up to 8pp return diff
    min_val_contrast=20,   # at least 20 pctile points apart
    top_n=30
)

print("\nWINNER PAIRS:")
winner_pairs = generate_pairs(
    winners, "Winners", 
    max_ret_diff=10.0,     # allow up to 10pp (wider range)
    min_val_contrast=20,
    top_n=30
)

In [ ]:
# ── Also try with industry-level percentiles ──
print("LOSER PAIRS (industry percentile):")
loser_pairs_ind = generate_pairs(
    losers, "Losers-Ind", 
    max_ret_diff=8.0,
    min_val_contrast=20,
    pctile_col='pb_pctile_industry',
    top_n=30
)

print("\nWINNER PAIRS (industry percentile):")
winner_pairs_ind = generate_pairs(
    winners, "Winners-Ind", 
    max_ret_diff=10.0,
    min_val_contrast=20,
    pctile_col='pb_pctile_industry',
    top_n=30
)

## 7. Display Top Candidate Pairs

In [ ]:
def display_pairs(df_pairs, title):
    if df_pairs.empty:
        print(f"\n{title}: No pairs found.")
        return
    
    show_cols = [
        'tic_low', 'name_low', 'ret_low', 'pb_pctile_low', 'mcap_bn_low', 'div_low',
        'tic_high', 'name_high', 'ret_high', 'pb_pctile_high', 'mcap_bn_high', 'div_high',
        'ret_diff_pp', 'val_contrast', 'mcap_diff_log', 'chart_corr', 'score'
    ]
    show_cols = [c for c in show_cols if c in df_pairs.columns]
    
    print(f"\n{'='*100}")
    print(f"{title}")
    print(f"{'='*100}")
    display(df_pairs[show_cols].head(15))

display_pairs(loser_pairs, "TOP LOSER PAIRS (Sector P/B Percentile)")
display_pairs(winner_pairs, "TOP WINNER PAIRS (Sector P/B Percentile)")
display_pairs(loser_pairs_ind, "TOP LOSER PAIRS (Industry P/B Percentile)")
display_pairs(winner_pairs_ind, "TOP WINNER PAIRS (Industry P/B Percentile)")

## 8. Familiar Anchors

In [ ]:
# ── Familiar anchor candidates ──
# Criteria: very large cap, well-known, ideally mid-range valuation
ANCHOR_CANDIDATES = ['MSFT', 'CSCO', 'AAPL', 'INTC', 'IBM', 'ORCL', 'DELL', 'HPQ', 'ADBE', 'CRM']

anchor_df = master[master['tic'].isin(ANCHOR_CANDIDATES)].copy()

if not anchor_df.empty:
    print("Familiar anchor candidates:")
    display(anchor_df[['tic', 'company_name', 'ret_365d_pct', 'pb_ratio', 'pb_pctile_sector', 
                       'val_sector', 'marketcap_bn', 'div_yield_pct', 'industry_yf']]
            .sort_values('marketcap_bn', ascending=False))
else:
    print("No anchor candidates found in master table. Check tickers.")
    # Show what we have that's very large cap
    big = master[master['marketcap_bn'] >= 100].sort_values('marketcap_bn', ascending=False)
    print(f"\nLargest stocks in our universe ({len(big)} with mcap >= $100B):")
    display(big[['tic', 'company_name', 'ret_365d_pct', 'pb_pctile_sector', 'val_sector', 'marketcap_bn']].head(20))

## 9. Visual Comparison of Top Pairs

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

def plot_pair(tic1, tic2, daily_series_dict, title=""):
    """Plot normalized return series for a pair of stocks."""
    if tic1 not in daily_series_dict or tic2 not in daily_series_dict:
        print(f"Cannot plot: {tic1} or {tic2} not in daily series.")
        return
    
    s1 = daily_series_dict[tic1]
    s2 = daily_series_dict[tic2]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    
    # Left: normalized return comparison
    ax1.plot(s1['Date'], s1['norm_ret'] * 100, label=tic1, linewidth=1.5)
    ax1.plot(s2['Date'], s2['norm_ret'] * 100, label=tic2, linewidth=1.5)
    ax1.axhline(0, color='gray', linestyle='--', linewidth=0.5)
    ax1.set_ylabel('Cumulative Return (%)')
    ax1.set_title(f'{title} — Normalized Returns')
    ax1.legend()
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %y'))
    ax1.tick_params(axis='x', rotation=45)
    
    # Right: raw price (rebased to 100)
    p1 = s1['Close'] / s1['Close'].iloc[0] * 100
    p2 = s2['Close'] / s2['Close'].iloc[0] * 100
    ax2.plot(s1['Date'], p1, label=tic1, linewidth=1.5)
    ax2.plot(s2['Date'], p2, label=tic2, linewidth=1.5)
    ax2.axhline(100, color='gray', linestyle='--', linewidth=0.5)
    ax2.set_ylabel('Price (rebased to 100)')
    ax2.set_title(f'{title} — Price Charts (rebased)')
    ax2.legend()
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %y'))
    ax2.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()


# Plot top loser pairs
print("=" * 60)
print("TOP LOSER PAIR CHARTS")
print("=" * 60)
if not loser_pairs.empty:
    for i in range(min(5, len(loser_pairs))):
        row = loser_pairs.iloc[i]
        plot_pair(
            row['tic_low'], row['tic_high'], daily_series,
            title=f"Loser #{i+1}: {row['tic_low']} (Low PB) vs {row['tic_high']} (High PB)"
        )

# Plot top winner pairs
print("\n" + "=" * 60)
print("TOP WINNER PAIR CHARTS")
print("=" * 60)
if not winner_pairs.empty:
    for i in range(min(5, len(winner_pairs))):
        row = winner_pairs.iloc[i]
        plot_pair(
            row['tic_low'], row['tic_high'], daily_series,
            title=f"Winner #{i+1}: {row['tic_low']} (Low PB) vs {row['tic_high']} (High PB)"
        )

## 10. Summary Dashboard

In [ ]:
# ── Print a summary of the full universe ──
print("FULL UNIVERSE SUMMARY")
print("=" * 60)
print(f"Total IT-sector stocks: {len(master)}")
print(f"\nReturn ranges:")
for name, (lo, hi) in [('Losers', LOSER_RANGE), ('Winners', WINNER_RANGE)]:
    n = len(master[(master['ret_365d_pct'] >= lo) & (master['ret_365d_pct'] <= hi)])
    print(f"  {name} ({lo}% to {hi}%): {n} stocks")

print(f"\n{'='*60}")
print("SELECTED PAIR CANDIDATES")
print(f"{'='*60}")

# Show concise top-3 for each
for name, df_p in [('Loser (Sector)', loser_pairs), ('Winner (Sector)', winner_pairs),
                   ('Loser (Industry)', loser_pairs_ind), ('Winner (Industry)', winner_pairs_ind)]:
    print(f"\n── {name} ──")
    if df_p.empty:
        print("  No pairs.")
        continue
    for i in range(min(3, len(df_p))):
        r = df_p.iloc[i]
        print(f"  #{i+1}: {r['tic_low']} (PB pctile={r['pb_pctile_low']:.0f}, ret={r['ret_low']:.1f}%, mcap=${r.get('mcap_bn_low',0):.0f}B) "
              f"vs {r['tic_high']} (PB pctile={r['pb_pctile_high']:.0f}, ret={r['ret_high']:.1f}%, mcap=${r.get('mcap_bn_high',0):.0f}B) "
              f"| Δret={r['ret_diff_pp']:.1f}pp, ΔPB={r['val_contrast']:.0f}pctile, chart_r={r['chart_corr']:.2f}, score={r['score']:.2f}")

## 11. Manual Deep-Dive: Explore Specific Candidates

Use this cell to explore specific tickers you're considering. Adjust the list below.

In [ ]:
# ── Deep dive into specific candidates ──
# Edit this list based on what looks promising above
EXPLORE = ['MSFT', 'CSCO']  # add your candidates here

for tic in EXPLORE:
    row = master[master['tic'] == tic]
    if row.empty:
        print(f"\n{tic}: Not found in master table")
        continue
    r = row.iloc[0]
    print(f"\n{'='*50}")
    print(f"{tic} — {r.get('company_name', r.get('conm', ''))}")
    print(f"{'='*50}")
    print(f"  Industry (YF):  {r.get('industry_yf', 'N/A')}")
    print(f"  GICS Industry:  {r.get('gind', 'N/A')}")
    print(f"  365d Return:    {r['ret_365d_pct']:.1f}%")
    print(f"  P/B Ratio:      {r['pb_ratio']:.2f}")
    print(f"  P/B Pctile (S): {r['pb_pctile_sector']:.0f}th")
    print(f"  P/B Pctile (I): {r['pb_pctile_industry']:.0f}th")
    print(f"  Valuation:      {r['val_sector']} (sector) / {r['val_industry']} (industry)")
    print(f"  Market Cap:     ${r.get('marketcap_bn', 0):.1f}B")
    print(f"  Div Yield:      {r.get('div_yield_pct', 0):.2f}%")
    print(f"  Current Price:  ${r.get('price_current', 0):.2f}")
    
    # Plot price chart if available
    if tic in daily_series:
        s = daily_series[tic]
        fig, ax = plt.subplots(figsize=(8, 3))
        ax.plot(s['Date'], s['Close'], linewidth=1.5, color='steelblue')
        ax.set_title(f"{tic} — 365-day Price")
        ax.set_ylabel('Price ($)')
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %y'))
        ax.tick_params(axis='x', rotation=45)
        plt.tight_layout()
        plt.show()

## 12. Export Final Selection (fill in after review)

In [ ]:
# ── Fill in your final selection after reviewing the candidates above ──
FINAL_SELECTION = {
    'loser_low_pb':  'XXX',   # Loser pair, low valuation
    'loser_high_pb': 'XXX',   # Loser pair, high valuation
    'winner_low_pb':  'XXX',  # Winner pair, low valuation
    'winner_high_pb': 'XXX',  # Winner pair, high valuation
    'anchor_1': 'MSFT',
    'anchor_2': 'CSCO',
}

# Show final comparison
all_final = [v for v in FINAL_SELECTION.values() if v != 'XXX']
if all_final:
    final_df = master[master['tic'].isin(all_final)].copy()
    final_df['role'] = final_df['tic'].map({v: k for k, v in FINAL_SELECTION.items()})
    display(final_df[['role', 'tic', 'company_name', 'industry_yf', 'ret_365d_pct', 
                      'pb_ratio', 'pb_pctile_sector', 'pb_pctile_industry',
                      'val_sector', 'marketcap_bn', 'div_yield_pct']].set_index('role'))
    
    # Plot all final selections together
    if len([t for t in all_final if t in daily_series]) >= 2:
        fig, ax = plt.subplots(figsize=(12, 5))
        for tic in all_final:
            if tic in daily_series:
                s = daily_series[tic]
                ax.plot(s['Date'], s['norm_ret'] * 100, label=tic, linewidth=1.5)
        ax.axhline(0, color='gray', linestyle='--', linewidth=0.5)
        ax.set_ylabel('Cumulative Return (%)')
        ax.set_title('Final Selection — 365-day Normalized Returns')
        ax.legend()
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %y'))
        ax.tick_params(axis='x', rotation=45)
        plt.tight_layout()
        plt.show()